# 05 — Movie Rating Prediction

Goal: predict the movie `score` from information available in the movie record.

To reduce leakage, this notebook does **not** use:
- gross revenue
- profit
- ROI

We also avoid `votes` because votes are an audience-response variable that becomes available after release.

Models:
- DummyRegressor baseline
- RandomForestRegressor

Metrics:
- MAE
- RMSE
- R²

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

movies = pd.read_csv("../data/processed/movies_cleaned.csv")
movies = movies.dropna(how="all").reset_index(drop=True)

for col in ["year", "score", "runtime", "release_year"]:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

movies["release_month"] = movies["release_month"].fillna("Unknown")
movies["genre"] = movies["genre"].fillna("Unknown")
movies["rating"] = movies["rating"].fillna("Unknown")
movies["country"] = movies["country"].fillna("Unknown")

features = [
    "year", "runtime", "release_year",
    "genre", "rating", "country", "release_month"
]

target = "score"

data = movies[features + [target]].dropna(subset=[target]).copy()

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = ["year", "runtime", "release_year"]
categorical_features = ["genre", "rating", "country", "release_month"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline MAE:", round(mean_absolute_error(y_test, baseline_pred), 4))
print("Baseline RMSE:", round(np.sqrt(mean_squared_error(y_test, baseline_pred)), 4))
print("Baseline R²:", round(r2_score(y_test, baseline_pred), 4))

In [ ]:
rating_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

rating_model.fit(X_train, y_train)
pred = rating_model.predict(X_test)

rating_metrics = {
    "MAE": mean_absolute_error(y_test, pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    "R2": r2_score(y_test, pred)
}

pd.Series(rating_metrics)

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Random Forest"],
    "MAE": [
        mean_absolute_error(y_test, baseline_pred),
        mean_absolute_error(y_test, pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, baseline_pred)),
        np.sqrt(mean_squared_error(y_test, pred))
    ],
    "R2": [
        r2_score(y_test, baseline_pred),
        r2_score(y_test, pred)
    ]
})

comparison

In [ ]:
results = pd.DataFrame({
    "actual_score": y_test.values,
    "predicted_score": pred
})

results["error"] = results["actual_score"] - results["predicted_score"]

results.head(20)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.scatterplot(data=results, x="actual_score", y="predicted_score", alpha=0.5)
plt.plot([2, 10], [2, 10], linestyle="--")
plt.title("Actual vs Predicted Movie Score")
plt.xlabel("Actual Score")
plt.ylabel("Predicted Score")
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(rating_model, "../models/rating_model.pkl")
print("Saved: ../models/rating_model.pkl")

## Interpretation

Use the metrics to explain whether the model improves on the baseline. A lower MAE/RMSE and a higher R² indicate better predictive performance on the held-out test set.

Do not claim that the model can perfectly predict audience ratings. The target is subjective and the dataset contains limited metadata.